# dtsupply.us Scraper — API-based (fast)

Unlike vertexnetworking, this site loads data through a JSON API instead of plain HTML, so this scraper calls the API directly:
- `get-all-categories` -> list of categories
- `get-products-by-category?id=X` -> full product list per category (already includes name, price, image, sku, stock status — no extra per-product calls needed)

Columns: `category, title, description, image, price, sku, in_stock, product_id`

- **description**: the API has no separate long description field, so this is built from the product's `specs` (e.g. "Form Factor: Internal; Product Type: SAS Controller"). Often empty if a product has no specs.
- **price**: if a product has no real price (0 or missing), the cell says `Get a Quote for price` instead of a false $0.00.
- **sku**: acts as the part number.

Run the pip install cell, then the main cell. It's resumable and saves incrementally — safe to stop and rerun anytime.

In [3]:
!pip install requests

In [4]:
#!/usr/bin/env python3
"""
Scraper for dtsupply.us — API-based (much faster than HTML scraping).

This site loads its data from a JSON API instead of plain HTML, so instead
of parsing web pages we call the API directly:

    1. GET https://api.dtsupply.us/api/Products/get-all-categories
       -> list of {category_id, name}

    2. For each category_id:
       GET https://api.dtsupply.us/api/Products/get-products-by-category?id=X
       -> list of products, each already containing:
          id, sku, name, price, isInStock, imageUrl, productUrl, specs

No HTML parsing needed at all — this returns clean, structured data
directly.

Requirements:
    pip install requests

Usage:
    python scrape_dtsupply.py

Output:
    dtsupply_products.csv   (written to as it goes)

Resumable: if you stop and rerun, it skips products (by their numeric id)
already present in the CSV.
"""

import csv
import os
import time

import requests

BASE_API = "https://api.dtsupply.us/api/Products"
CATEGORIES_URL = f"{BASE_API}/get-all-categories"
PRODUCTS_BY_CATEGORY_URL = f"{BASE_API}/get-products-by-category"

OUTPUT_CSV = "dtsupply_products.csv"
REQUEST_DELAY = 0.3   # seconds between API calls, be polite to their server
TIMEOUT = 20
PROGRESS_EVERY = 50

FIELDNAMES = [
    "category", "title", "description", "image", "price",
    "sku", "in_stock", "product_id",
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
}

session = requests.Session()
session.headers.update(HEADERS)


def get_json(url, params=None):
    try:
        resp = session.get(url, params=params, timeout=TIMEOUT)
        resp.raise_for_status()
        time.sleep(REQUEST_DELAY)
        return resp.json()
    except requests.RequestException as e:
        print(f"  [warn] failed to fetch {url} (params={params}): {e}")
        return None


def format_price(price):
    """Some products have no real price (call for pricing). Treat 0/None
    as a quote-required item instead of showing a misleading $0.00."""
    if price is None or price == 0:
        return "Get a Quote for price"
    return f"${price:,.2f}"


def specs_to_text(specs):
    """Flatten the specs dict (if present) into one readable line, used as
    the description since the API has no separate long description field."""
    if not specs or not isinstance(specs, dict):
        return ""
    return "; ".join(f"{k}: {v}" for k, v in specs.items())


def load_already_done():
    """Product ids already written to the CSV, so reruns can resume."""
    done = set()
    if os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row.get("product_id"):
                    done.add(str(row["product_id"]))
    return done


def open_csv_writer():
    file_exists = os.path.exists(OUTPUT_CSV)
    f = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    if not file_exists:
        writer.writeheader()
        f.flush()
    return f, writer


def main():
    already_done = load_already_done()
    if already_done:
        print(f"Resuming — {len(already_done)} products already in {OUTPUT_CSV}, will skip those.")

    csv_file, writer = open_csv_writer()
    scraped_count = len(already_done)

    try:
        print("Fetching category list...")
        categories = get_json(CATEGORIES_URL)
        if not categories:
            raise SystemExit("Could not load categories — aborting.")
        print(f"Found {len(categories)} categories.")

        for cat_index, cat in enumerate(categories, start=1):
            cat_id = cat.get("category_id")
            cat_name = cat.get("name", f"category-{cat_id}")
            print(f"\n[{cat_index}/{len(categories)}] {cat_name} (id={cat_id})")

            products = get_json(PRODUCTS_BY_CATEGORY_URL, params={"id": cat_id})
            if not products:
                print("  (no products or fetch failed, skipping)")
                continue

            print(f"  {len(products)} products in this category")

            for p in products:
                product_id = str(p.get("id", ""))
                if product_id and product_id in already_done:
                    continue

                row = {
                    "category": cat_name,
                    "title": p.get("name", ""),
                    "description": specs_to_text(p.get("specs")),
                    "image": p.get("imageUrl", ""),
                    "price": format_price(p.get("price")),
                    "sku": p.get("sku", ""),
                    "in_stock": p.get("isInStock", ""),
                    "product_id": product_id,
                }
                writer.writerow(row)
                csv_file.flush()
                already_done.add(product_id)
                scraped_count += 1

                if scraped_count % PROGRESS_EVERY == 0:
                    print(f"  >>> progress: {scraped_count} products saved so far")

    except KeyboardInterrupt:
        print("\n[stopped by user] Progress so far is safely saved in the CSV.")
    finally:
        csv_file.close()

    print(f"\nTotal products in {OUTPUT_CSV}: {scraped_count}")
    print("Done (or safely stopped — rerun the script anytime to resume).")


if __name__ == "__main__":
    main()


Fetching category list...
Found 18 categories.

[1/18] Audio Components (id=11)
  168 products in this category
  >>> progress: 50 products saved so far
  >>> progress: 100 products saved so far
  >>> progress: 150 products saved so far

[2/18] Cables (id=14)
  6842 products in this category
  >>> progress: 200 products saved so far
  >>> progress: 250 products saved so far
  >>> progress: 300 products saved so far
  >>> progress: 350 products saved so far
  >>> progress: 400 products saved so far
  >>> progress: 450 products saved so far
  >>> progress: 500 products saved so far
  >>> progress: 550 products saved so far
  >>> progress: 600 products saved so far
  >>> progress: 650 products saved so far
  >>> progress: 700 products saved so far
  >>> progress: 750 products saved so far
  >>> progress: 800 products saved so far
  >>> progress: 850 products saved so far
  >>> progress: 900 products saved so far
  >>> progress: 950 products saved so far
  >>> progress: 1000 products saved